In [2]:
"""
CRM Streaming Pipeline — Offline / Local Mode
Simulates HubSpot & Salesforce Kafka events, validates schema + data quality,
deduplicates, enriches, and writes partitioned NDJSON files to local disk.

"""

import json
import logging
import os
import random
import re
import string
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, FrozenSet, Iterator, List, Optional, Tuple

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)
logger = logging.getLogger("crm_pipeline")


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("crm_output")          # all files written here
FLUSH_BATCH_SIZE = 10                    # write to disk every N valid records
TOTAL_MOCK_EVENTS_PER_SOURCE = 40        # how many mock events to generate
DUPLICATE_RATE = 0.15                    # 15 % of events are intentional duplicates
INVALID_RATE = 0.10                      # 10 % of events have bad data


# ============================================================
# SCHEMAS & VALIDATION CONSTANTS
# ============================================================

HUBSPOT_REQUIRED_FIELDS: FrozenSet[str] = frozenset({
    "event_id", "event_type", "occurred_at", "object_type", "object_id",
})
SALESFORCE_REQUIRED_FIELDS: FrozenSet[str] = frozenset({
    "event_id", "event_type", "created_date", "sobject_type", "record_id",
})

HUBSPOT_VALID_OBJECT_TYPES: FrozenSet[str] = frozenset({
    "contact", "company", "deal", "ticket", "product",
})
SALESFORCE_VALID_SOBJECT_TYPES: FrozenSet[str] = frozenset({
    "Lead", "Contact", "Account", "Opportunity", "Case", "Campaign",
})
HUBSPOT_VALID_EVENT_TYPES: FrozenSet[str] = frozenset({
    "contact.creation", "contact.propertyChange", "contact.deletion",
    "deal.creation", "deal.propertyChange", "deal.deletion",
    "company.creation", "company.propertyChange",
})
SALESFORCE_VALID_EVENT_TYPES: FrozenSet[str] = frozenset({
    "Created", "Updated", "Deleted", "Undeleted",
})

# Numeric sanity bounds: field_name -> (min, max)
HUBSPOT_NUMERIC_RANGES: Dict[str, Tuple[float, float]] = {
    "deal_amount": (0, 1_000_000_000),
    "num_associated_contacts": (0, 100_000),
}
SALESFORCE_NUMERIC_RANGES: Dict[str, Tuple[float, float]] = {
    "amount": (0, 1_000_000_000),
    "number_of_employees": (0, 10_000_000),
}

MAX_STRING_LENGTH: int = 10_000
ISO8601_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}")


# ============================================================
# VALIDATION
# ============================================================

class ValidationError(Exception):
    """Raised when a record fails schema or data-quality checks."""


def _check_required_fields(record: Dict[str, Any], required: FrozenSet[str], source: str) -> None:
    missing = required - record.keys()
    if missing:
        raise ValidationError(f"[{source}] Missing required fields: {missing}")


def _check_string_lengths(record: Dict[str, Any], source: str) -> None:
    for key, val in record.items():
        if isinstance(val, str) and len(val) > MAX_STRING_LENGTH:
            raise ValidationError(
                f"[{source}] Field '{key}' exceeds max string length ({MAX_STRING_LENGTH})"
            )


def _check_numeric_ranges(
    record: Dict[str, Any],
    ranges: Dict[str, Tuple[float, float]],
    source: str,
) -> None:
    for fname, (lo, hi) in ranges.items():
        val = record.get(fname)
        if val is None:
            continue
        if not isinstance(val, (int, float)):
            raise ValidationError(
                f"[{source}] Field '{fname}' must be numeric, got {type(val).__name__}"
            )
        if not lo <= val <= hi:
            raise ValidationError(
                f"[{source}] Field '{fname}'={val} out of range [{lo}, {hi}]"
            )


def validate_hubspot(record: Dict[str, Any]) -> None:
    """Schema + quality validation for a HubSpot event record."""
    _check_required_fields(record, HUBSPOT_REQUIRED_FIELDS, "HubSpot")

    if record["event_type"] not in HUBSPOT_VALID_EVENT_TYPES:
        raise ValidationError(f"[HubSpot] Unknown event_type: '{record['event_type']}'")

    if record["object_type"] not in HUBSPOT_VALID_OBJECT_TYPES:
        raise ValidationError(f"[HubSpot] Unknown object_type: '{record['object_type']}'")

    occurred_at = record.get("occurred_at")
    if not isinstance(occurred_at, (int, float)) or occurred_at < 0:
        raise ValidationError(
            f"[HubSpot] 'occurred_at' must be non-negative epoch ms, got: {occurred_at}"
        )

    _check_numeric_ranges(record, HUBSPOT_NUMERIC_RANGES, "HubSpot")
    _check_string_lengths(record, "HubSpot")


def validate_salesforce(record: Dict[str, Any]) -> None:
    """Schema + quality validation for a Salesforce event record."""
    _check_required_fields(record, SALESFORCE_REQUIRED_FIELDS, "Salesforce")

    if record["event_type"] not in SALESFORCE_VALID_EVENT_TYPES:
        raise ValidationError(f"[Salesforce] Unknown event_type: '{record['event_type']}'")

    if record["sobject_type"] not in SALESFORCE_VALID_SOBJECT_TYPES:
        raise ValidationError(f"[Salesforce] Unknown sobject_type: '{record['sobject_type']}'")

    created_date = record.get("created_date", "")
    if not isinstance(created_date, str) or not ISO8601_RE.match(created_date):
        raise ValidationError(
            f"[Salesforce] 'created_date' must be ISO-8601, got: '{created_date}'"
        )

    _check_numeric_ranges(record, SALESFORCE_NUMERIC_RANGES, "Salesforce")
    _check_string_lengths(record, "Salesforce")


# ============================================================
# DATA QUALITY METRICS
# ============================================================

class QualityMetrics:
    """Tracks per-source pipeline counters and prints a summary at the end."""

    def __init__(self, source: str) -> None:
        self.source = source
        self._counts: Dict[str, int] = defaultdict(int)

    def inc(self, key: str, n: int = 1) -> None:
        self._counts[key] += n

    def summary(self) -> Dict[str, int]:
        return dict(self._counts)

    def log_summary(self) -> None:
        s = self._counts
        logger.info(
            "[%s] FINAL METRICS | consumed=%d  valid=%d  invalid=%d  duplicates=%d  written=%d",
            self.source,
            s.get("consumed", 0),
            s.get("valid", 0),
            s.get("invalid", 0),
            s.get("duplicates", 0),
            s.get("written", 0),
        )


# ============================================================
# LOCAL DISK WRITER  (replaces MinIO)
# ============================================================

class LocalDiskWriter:
    """
    Buffers validated records and flushes them to local NDJSON files.
    Output path mirrors MinIO partitioning: <prefix>/year=/month=/day=/hour=/part-<uuid>.json
    """

    def __init__(self, base_dir: Path, prefix: str, metrics: QualityMetrics) -> None:
        self._base = base_dir
        self._prefix = prefix
        self._metrics = metrics
        self._buffer: List[str] = []

    def _partition_path(self) -> Path:
        now = datetime.now(tz=timezone.utc)
        folder = (
            self._base
            / self._prefix
            / f"year={now.year}"
            / f"month={now.month:02d}"
            / f"day={now.day:02d}"
            / f"hour={now.hour:02d}"
        )
        folder.mkdir(parents=True, exist_ok=True)
        return folder / f"part-{uuid.uuid4().hex}.json"

    def write(self, record_json: str) -> None:
        self._buffer.append(record_json)
        if len(self._buffer) >= FLUSH_BATCH_SIZE:
            self.flush()

    def flush(self) -> None:
        if not self._buffer:
            return
        path = self._partition_path()
        with path.open("w", encoding="utf-8") as fh:
            fh.write("\n".join(self._buffer) + "\n")
        self._metrics.inc("written", len(self._buffer))
        logger.info("[%s] Flushed %d records -> %s", self._metrics.source, len(self._buffer), path)
        self._buffer.clear()


# ============================================================
# MOCK DATA GENERATORS  (replaces Kafka)
# ============================================================

def _rand_id() -> str:
    return uuid.uuid4().hex


def _rand_str(length: int = 8) -> str:
    return "".join(random.choices(string.ascii_lowercase, k=length))


def generate_hubspot_event(force_invalid: bool = False, reuse_id: Optional[str] = None) -> Dict[str, Any]:
    """Produces a realistic HubSpot event dict; optionally malformed for testing."""
    event_id = reuse_id or _rand_id()
    record: Dict[str, Any] = {
        "event_id":               event_id,
        "event_type":             random.choice(list(HUBSPOT_VALID_EVENT_TYPES)),
        "occurred_at":            int(datetime.now(tz=timezone.utc).timestamp() * 1000),
        "object_type":            random.choice(list(HUBSPOT_VALID_OBJECT_TYPES)),
        "object_id":              random.randint(1000, 9999),
        "portal_id":              random.randint(100, 999),
        "deal_amount":            round(random.uniform(500, 500_000), 2),
        "num_associated_contacts": random.randint(0, 50),
        "properties": {
            "firstname": _rand_str(),
            "lastname":  _rand_str(),
            "email":     f"{_rand_str()}@example.com",
        },
    }
    if force_invalid:
        fault = random.choice(["missing_field", "bad_event_type", "bad_numeric"])
        if fault == "missing_field":
            del record["event_type"]
        elif fault == "bad_event_type":
            record["event_type"] = "UNKNOWN_TYPE"
        elif fault == "bad_numeric":
            record["deal_amount"] = -999
    return record


def generate_salesforce_event(force_invalid: bool = False, reuse_id: Optional[str] = None) -> Dict[str, Any]:
    """Produces a realistic Salesforce event dict; optionally malformed for testing."""
    event_id = reuse_id or _rand_id()
    record: Dict[str, Any] = {
        "event_id":           event_id,
        "event_type":         random.choice(list(SALESFORCE_VALID_EVENT_TYPES)),
        "created_date":       datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        "sobject_type":       random.choice(list(SALESFORCE_VALID_SOBJECT_TYPES)),
        "record_id":          _rand_id().upper()[:18],
        "org_id":             _rand_id().upper()[:15],
        "replay_id":          random.randint(1, 100_000),
        "amount":             round(random.uniform(1000, 1_000_000), 2),
        "number_of_employees": random.randint(1, 50_000),
        "payload": {
            "Name":  f"{_rand_str()} Corp",
            "Stage": random.choice(["Prospecting", "Negotiation", "Closed Won"]),
        },
    }
    if force_invalid:
        fault = random.choice(["missing_field", "bad_event_type", "bad_date"])
        if fault == "missing_field":
            del record["sobject_type"]
        elif fault == "bad_event_type":
            record["event_type"] = "INVALID"
        elif fault == "bad_date":
            record["created_date"] = "not-a-date"
    return record


def mock_kafka_stream(
    generator: Callable,
    total: int,
    duplicate_rate: float,
    invalid_rate: float,
) -> Iterator[str]:
    """
    Yields JSON strings simulating messages from a Kafka topic.
    Mixes valid, invalid, and duplicate records according to given rates.
    """
    seen_ids: List[str] = []
    for _ in range(total):
        is_invalid   = random.random() < invalid_rate
        is_duplicate = bool(seen_ids) and random.random() < duplicate_rate

        reuse_id = random.choice(seen_ids) if is_duplicate else None
        record   = generator(force_invalid=is_invalid, reuse_id=reuse_id)

        if not is_duplicate and "event_id" in record:
            seen_ids.append(record["event_id"])

        yield json.dumps(record, ensure_ascii=False)


# ============================================================
# PIPELINE PROCESSOR
# ============================================================

class CRMProcessor:
    """
    Consumes a stream of raw JSON strings through the full pipeline:
      parse -> validate -> deduplicate -> enrich -> buffer -> flush to disk
    """

    def __init__(
        self,
        source_name: str,
        validator: Callable[[Dict[str, Any]], None],
        writer: LocalDiskWriter,
        metrics: QualityMetrics,
    ) -> None:
        self._source   = source_name
        self._validate = validator
        self._writer   = writer
        self._metrics  = metrics
        self._seen_ids: set = set()

    def _parse(self, raw: str) -> Optional[Dict[str, Any]]:
        try:
            return json.loads(raw)
        except json.JSONDecodeError as exc:
            logger.warning("[%s] Unparseable JSON dropped: %.120s | %s", self._source, raw, exc)
            return None

    def _enrich(self, record: Dict[str, Any]) -> Dict[str, Any]:
        record["_pipeline_source"] = self._source
        record["_ingested_at"]     = datetime.now(tz=timezone.utc).isoformat()
        return record

    def _is_duplicate(self, record: Dict[str, Any]) -> bool:
        eid = record.get("event_id")
        if eid and eid in self._seen_ids:
            self._metrics.inc("duplicates")
            return True
        if eid:
            self._seen_ids.add(eid)
        return False

    def process(self, stream: Iterator[str]) -> None:
        for raw in stream:
            self._metrics.inc("consumed")

            record = self._parse(raw)
            if record is None:
                self._metrics.inc("invalid")
                continue

            try:
                self._validate(record)
            except ValidationError as exc:
                self._metrics.inc("invalid")
                logger.warning("[%s] Validation failed - dropped | %s", self._source, exc)
                continue

            if self._is_duplicate(record):
                logger.debug("[%s] Duplicate event_id dropped: %s", self._source, record.get("event_id"))
                continue

            self._metrics.inc("valid")
            enriched = self._enrich(record)
            self._writer.write(json.dumps(enriched, ensure_ascii=False))

        self._writer.flush()
        self._metrics.log_summary()


# ============================================================
# ENTRYPOINT
# ============================================================

def run_pipeline() -> Dict[str, Dict[str, int]]:
    output_dir = OUTPUT_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Output directory: %s", output_dir.resolve())

    sources = [
        ("HubSpot",    generate_hubspot_event,    validate_hubspot,    "hubspot"),
        ("Salesforce", generate_salesforce_event, validate_salesforce, "salesforce"),
    ]

    all_metrics: Dict[str, Dict[str, int]] = {}

    for source_name, generator, validator, prefix in sources:
        logger.info("--- Processing %s ---", source_name)
        metrics   = QualityMetrics(source_name)
        writer    = LocalDiskWriter(output_dir, prefix, metrics)
        processor = CRMProcessor(source_name, validator, writer, metrics)

        stream = mock_kafka_stream(
            generator=generator,
            total=TOTAL_MOCK_EVENTS_PER_SOURCE,
            duplicate_rate=DUPLICATE_RATE,
            invalid_rate=INVALID_RATE,
        )
        processor.process(stream)
        all_metrics[source_name] = metrics.summary()

    logger.info("Pipeline complete. Files written to: %s", output_dir.resolve())
    return all_metrics


# ============================================================
# RUN
# ============================================================

metrics = run_pipeline()

print("\n========== PIPELINE SUMMARY ==========")
for source, counts in metrics.items():
    print(f"\n{source}")
    for k, v in sorted(counts.items()):
        print(f"  {k:<12} {v}")

print("\nOutput files:")
for f in sorted(OUTPUT_DIR.rglob("*.json")):
    size = f.stat().st_size
    print(f"  {f}  ({size} bytes)")

2026-05-15 08:27:24,761 [INFO] crm_pipeline - Output directory: /home/f0ffe0e4-1f12-4a6d-a4c8-7b1806fca8ad/crm_output
2026-05-15 08:27:24,762 [INFO] crm_pipeline - --- Processing HubSpot ---
2026-05-15 08:27:24,763 [WARNING] crm_pipeline - [HubSpot] Validation failed - dropped | [HubSpot] Missing required fields: {'event_type'}
2026-05-15 08:27:24,772 [INFO] crm_pipeline - [HubSpot] Flushed 10 records -> crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-8e158abe4634406eaf5947a742534d5c.json
2026-05-15 08:27:24,773 [WARNING] crm_pipeline - [HubSpot] Validation failed - dropped | [HubSpot] Field 'deal_amount'=-999 out of range [0, 1000000000]
2026-05-15 08:27:24,781 [INFO] crm_pipeline - [HubSpot] Flushed 10 records -> crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-d4d876a6c9744ea6be3fed592b866e43.json
2026-05-15 08:27:24,789 [INFO] crm_pipeline - [HubSpot] Flushed 10 records -> crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-655476a2515a4df09be815879a2f156c.


========== PIPELINE SUMMARY ==========

HubSpot
  consumed     40
  duplicates   8
  invalid      2
  valid        30
  written      30

Salesforce
  consumed     40
  duplicates   7
  invalid      2
  valid        31
  written      31

Output files:
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-029a4f7f5c214f1f84671e9727422d47.json  (4132 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-1e0acb7f0c344ae2ad0a162fbba6bdd3.json  (4127 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-2f380144671b470db3f868c1be4a000c.json  (4115 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-322a40481ef6484397ee1f09a2d7f6a9.json  (4129 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-63c7dae3f12144e0a9236cf8a7be034e.json  (413 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-655476a2515a4df09be815879a2f156c.json  (4128 bytes)
  crm_output/hubspot/year=2026/month=05/day=15/hour=08/part-6e0d665708ef474ab